# Explicabilidad para atribución de autoría con un *transformer* entrenado usando BERTViz

Este notebook está preparado para ser ejecutado en Google Colab puesto que se necesita GPU para la inferencia.

Hace lo siguiente:

- Carga un modelo `AutoModelForSequenceClassification` ya entrenado.
- Carga su tokenizer y sus etiquetas.
- Predice la autoría para un texto o documento largo.
- Divide el documento largo en *chunks*.
- Selecciona los *chunks* más informativos.
- Visualiza patrones de atención con **BERTViz**.

Este notebook sirve como complemento a `11_transformer_explainability_captum.ipynb`, no como sustituto.

## BERTViz

**BertViz** es una librería de visualización interactiva diseñada para analizar los mecanismos de atención en modelos basados en la arquitectura *transformer*, como BERT, GPT-2, T5, RoBERTa o modelos compatibles con Hugging Face. Su objetivo es facilitar la interpretación de cómo el modelo relaciona unas palabras con otras durante el procesamiento del texto, mostrando visualmente los pesos de atención entre tokens, capas y cabezas de atención.

En este *notebook*, BertViz se utiliza como una herramienta complementaria de explicabilidad para inspeccionar el comportamiento interno del modelo. Mientras que métodos como Captum permiten estimar qué palabras contribuyen más a una predicción concreta, BertViz permite observar cómo fluye la atención dentro del *transformer*, es decir, qué tokens atienden a otros tokens, qué cabezas capturan relaciones relevantes y cómo varían estos patrones entre capas. Esto puede ayudar a interpretar si el modelo está prestando atención a elementos estilísticos, léxicos o estructurales del texto que podrían estar relacionados con la atribución de autoría.

BertViz ofrece varias vistas de análisis, como la visualización por cabezas de atención, la vista de modelo y la vista de neuronas, proporcionando distintos niveles de granularidad para explorar el funcionamiento del *transformer*. Estas visualizaciones no explican directamente la predicción final como una atribución causal, pero sí proporcionan una ventana útil al mecanismo interno de atención del modelo, ayudando a contextualizar qué relaciones entre palabras han sido relevantes durante el procesamiento del texto.

**Repositorio de BERTViz:** <https://github.com/jessevig/bertviz>

In [1]:
!pip install -q transformers scikit-learn bertviz torch pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.5/157.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 96.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.6 MB/s eta 0:00:00


In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import json
from pathlib import Path
from typing import Any, Mapping, Optional, Sequence

import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, IFrame

from transformers import AutoModelForSequenceClassification, AutoTokenizer
from bertviz import head_view, model_view

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA disponible:", torch.cuda.is_available())
print("Dispositivo:", DEVICE)

CUDA disponible: True
Dispositivo: cuda


## Configuración

In [4]:
# Ruta local al directorio del modelo entrenado
MODEL_DIR = "/content/drive/MyDrive/outputs/authorship_transformer"

# Ruta local al texto a analizar
TEXT_PATH = Path("/content/drive/MyDrive/corpus/arthur_conan_doyle/acd-the_hound_of_the_baskervilles.txt")
# TEXT_PATH = Path("/content/drive/MyDrive/corpus/arthur_conan_doyle/acd-the_lost_world.txt")
PASTED_TEXT = None  # Podemos pegar aquí un fragmento de texto en lugar de usar una obra cargada

# Parámetros de clasificación por chunks
MAX_LENGTH = 128
STRIDE = 32
BATCH_SIZE = 8

# Parámetros para que BERTViz sea manejable
TOP_CHUNKS_TO_INSPECT = 1
MAX_TOKENS_FOR_BERTVIZ = 48
SAVE_BERTVIZ_HTML = True
SHOW_HEAD_VIEW = True
SHOW_MODEL_VIEW = True  # Activar solo si es necesario, es bastante más pesado

# Chunk concreto para inspección manual
MANUAL_CHUNK_ID = 0

# Carpeta donde se guardan los HTML externos
HTML_DIR = Path(MODEL_DIR) / "bertviz_html"
HTML_DIR.mkdir(parents=True, exist_ok=True)

print("HTML externos en:", HTML_DIR.resolve())

HTML externos en: /content/drive/MyDrive/outputs/authorship_transformer/bertviz_html


## Carga del modelo y tokenizer

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR,
    attn_implementation="eager",
)
model.to(DEVICE)
model.eval()

id2label = model.config.id2label
if isinstance(next(iter(id2label.keys())), str):
    id2label = {int(k): v for k, v in id2label.items()}
label2id = model.config.label2id

print("Modelo cargado desde:", MODEL_DIR)
print("Etiquetas:", id2label)
print("attn_implementation:", getattr(model.config, "_attn_implementation", "unknown"))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modelo cargado desde: /content/drive/MyDrive/outputs/authorship_transformer
Etiquetas: {0: 'anna_katharine_green', 1: 'arthur_conan_doyle', 2: 'arthur_morrison', 3: 'gilbert_keith_chesterton', 4: 'richard_austin_freeman', 5: 'wilkie_collins'}
attn_implementation: eager


## Funciones auxiliares

Dado que este *notebook* se ejecutará en Colab, es necesario definir algunas funciones auxiliares para cargar los datos y preparar el dataset para el entrenamiento. Estas funciones se han adaptado para funcionar con la estructura de archivos en Colab.

A diferencia de los *notebooks* anteriores, donde las utilidades estaban organizadas en módulos separados, aquí se incluyen directamente en el notebook para facilitar su ejecución sin necesidad de importar desde otros archivos.

In [6]:
# @title Funciones auxiliares
def get_text(text_path: Optional[Path] = None, pasted_text: Optional[object] = None) -> str:
    """Return text from a pasted value or a local text file.

    If `pasted_text` is provided and contains non-whitespace content after being
    converted to a string, that value is returned with surrounding whitespace
    removed. Otherwise, if `text_path` is provided and points to an existing file,
    the file is read as UTF-8 and returned with surrounding whitespace removed.

    Args:
        text_path: Optional path to a local text file.
        pasted_text: Optional value containing pasted text. The value is converted
            to a string before validation and return.

    Returns:
        The extracted text with leading and trailing whitespace removed.
    """
    if pasted_text is not None and str(pasted_text).strip():
        return str(pasted_text).strip()
    if text_path is not None and Path(text_path).exists():
        return Path(text_path).read_text(encoding="utf-8", errors="replace").strip()
    raise FileNotFoundError("No se encontró texto. Ajusta TEXT_PATH o asigna contenido a PASTED_TEXT.")


def chunk_document(text: str, max_length: int = 256, stride: int = 64) -> list[dict[str, list[int]]]:
    """Tokenize a document into overlapping chunks.

    The function uses the global `tokenizer` object to split `text` into token
    chunks with optional overlap. Each returned chunk contains token IDs and an
    attention mask suitable for model input.

    Args:
        text: Document text to tokenize.
        max_length: Maximum number of tokens per chunk, including special tokens.
        stride: Number of overlapping tokens between consecutive chunks.

    Returns:
        A list of dictionaries. Each dictionary contains:
            input_ids: Token IDs for a chunk.
            attention_mask: Attention mask for the corresponding chunk.
    """
    enc = tokenizer(
        text,
        add_special_tokens=True,
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        return_token_type_ids=False,
    )
    return [
        {"input_ids": enc["input_ids"][i], "attention_mask": enc["attention_mask"][i]}
        for i in range(len(enc["input_ids"]))
    ]


def decode_chunk(chunk: Mapping[str, Sequence[int]]) -> str:
    """Decode a tokenized chunk into text.

    The function uses the global `tokenizer` object to decode the `input_ids`
    sequence from `chunk`, skipping special tokens.

    Args:
        chunk: Mapping containing an `input_ids` sequence of token IDs.

    Returns:
        The decoded text with special tokens omitted.
    """
    return tokenizer.decode(chunk["input_ids"], skip_special_tokens=True)


@torch.no_grad()
def predict_chunks(
    chunks: Sequence[Mapping[str, Sequence[int]]],
    batch_size: int = 8,
) -> tuple[np.ndarray, np.ndarray]:
    """Predict classes for tokenized chunks in batches.

    The function pads each batch to the longest chunk in that batch, runs the
    global `model` without gradient tracking, and returns class probabilities and
    predicted class indices for all chunks.

    Args:
        chunks: Tokenized chunks to classify. Each chunk must contain `input_ids`
            and `attention_mask` sequences.
        batch_size: Number of chunks to process per model call.

    Returns:
        A tuple containing:
            - Class probabilities with shape `(num_chunks, num_classes)`.
            - Predicted class indices with shape `(num_chunks,)`.
    """
    all_probs = []
    all_preds = []

    for start in range(0, len(chunks), batch_size):
        batch = chunks[start : start + batch_size]
        max_len = max(len(c["input_ids"]) for c in batch)

        input_ids = []
        attention_mask = []
        pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
        if pad_id is None:
            pad_id = 0

        for c in batch:
            ids = list(c["input_ids"])
            mask = list(c["attention_mask"])
            pad = max_len - len(ids)
            input_ids.append(ids + [pad_id] * pad)
            attention_mask.append(mask + [0] * pad)

        inputs = {
            "input_ids": torch.tensor(input_ids, device=DEVICE),
            "attention_mask": torch.tensor(attention_mask, device=DEVICE),
        }
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).detach().cpu().numpy()
        preds = np.argmax(probs, axis=1)
        all_probs.append(probs)
        all_preds.append(preds)

    return np.vstack(all_probs), np.concatenate(all_preds)


def predict_document(text: str, max_length: int = 256, stride: int = 64, batch_size: int = 8) -> dict[str, Any]:
    """Predict a document label by aggregating predictions from tokenized chunks.

    The document is split into overlapping chunks, each chunk is classified in
    batches, and the final document probabilities are computed by averaging the
    chunk-level probabilities across all chunks. The predicted document label is
    the label with the highest averaged probability.

    Args:
        text: Document text to classify.
        max_length: Maximum number of tokens per chunk, including special tokens.
        stride: Number of overlapping tokens between consecutive chunks.
        batch_size: Number of chunks to process per model inference batch.

    Returns:
        A dictionary containing:
            chunks: Tokenized chunks generated from the document.
            chunk_df: DataFrame with one row per chunk, including predicted label,
                confidence, token count, text preview, and per-label probabilities.
            doc_probs: Averaged document-level class probabilities.
            doc_pred_id: Predicted document class index.
            doc_pred_label: Predicted document class label.
    """
    chunks = chunk_document(text, max_length=max_length, stride=stride)
    probs, preds = predict_chunks(chunks, batch_size=batch_size)

    avg_probs = probs.mean(axis=0)
    pred = int(np.argmax(avg_probs))

    rows = []
    for i, p in enumerate(probs):
        row = {
            "chunk_id": i,
            "pred_label": id2label[int(preds[i])],
            "pred_confidence": float(np.max(p)),
            "n_tokens": len(chunks[i]["input_ids"]),
            "text_preview": decode_chunk(chunks[i])[:220].replace("\n", " "),
        }
        for j in range(len(p)):
            row[f"prob_{id2label[j]}"] = float(p[j])
        rows.append(row)

    return {
        "chunks": chunks,
        "chunk_df": pd.DataFrame(rows),
        "doc_probs": avg_probs,
        "doc_pred_id": pred,
        "doc_pred_label": id2label[pred],
    }


def pretty_probabilities(prob_vector: Sequence[float]) -> pd.DataFrame:
    """Format class probabilities as a sorted DataFrame.

    The function maps each probability in `prob_vector` to a label from the
    global `id2label` mapping, then returns the rows sorted by descending
    probability.

    Args:
        prob_vector: Sequence of class probabilities ordered by class index.

    Returns:
        A DataFrame with `label` and `probability` columns, sorted from highest
        to lowest probability.
    """
    rows = [{"label": id2label[i], "probability": float(p)} for i, p in enumerate(prob_vector)]
    return pd.DataFrame(rows).sort_values("probability", ascending=False).reset_index(drop=True)


def trim_chunk_for_viz(chunk: Mapping[str, Sequence[int]], max_tokens: int = 48) -> dict[str, Sequence[int]]:
    """Trim a tokenized chunk before visualization.

    The function keeps only the first `max_tokens` token IDs and attention mask
    values from a chunk. This is useful before computing attention visualizations
    for tools that can become slow or memory-intensive with long sequences.

    Args:
        chunk: Tokenized chunk containing `input_ids` and `attention_mask`
            sequences.
        max_tokens: Maximum number of tokens to keep from the beginning of the
            chunk.

    Returns:
        A dictionary containing trimmed `input_ids` and `attention_mask` lists.
    """
    return {
        "input_ids": list(chunk["input_ids"])[:max_tokens],
        "attention_mask": list(chunk["attention_mask"])[:max_tokens],
    }


@torch.no_grad()
def get_chunk_attention(chunk: Mapping[str, Sequence[int]]) -> dict[str, Any]:
    """Return attention tensors and prediction details for a tokenized chunk.

    The function runs the global `model` on a single tokenized chunk with
    attention outputs enabled. It returns the decoded token strings, attention
    tensors moved to CPU, class probabilities, and the predicted class label.

    Args:
        chunk: Tokenized chunk containing `input_ids` and `attention_mask`
            sequences.

    Returns:
        A dictionary containing:
            tokens: Token strings converted from `input_ids`.
            attentions: Tuple of attention tensors, one per model layer, moved to
                CPU.
            probs: Class probabilities for the chunk.
            pred_id: Predicted class index.
            pred_label: Predicted class label.
    """
    inputs = {
        "input_ids": torch.tensor([chunk["input_ids"]], device=DEVICE),
        "attention_mask": torch.tensor([chunk["attention_mask"]], device=DEVICE),
    }
    outputs = model(**inputs, output_attentions=True)

    if outputs.attentions is None:
        raise ValueError("El modelo no devolvió atenciones. Recárgalo con attn_implementation='eager'.")

    attentions = tuple(att.detach().cpu() for att in outputs.attentions)
    probs = torch.softmax(outputs.logits.detach().cpu(), dim=-1).numpy()[0]
    pred_id = int(np.argmax(probs))
    tokens = tokenizer.convert_ids_to_tokens(chunk["input_ids"])

    return {
        "tokens": tokens,
        "attentions": attentions,
        "probs": probs,
        "pred_id": pred_id,
        "pred_label": id2label[pred_id],
    }


def _html_data(obj: Any) -> str:
    """Extract HTML content from a BERTViz or IPython display object.

    The function checks common HTML-bearing attributes and methods used by
    display objects. It returns an empty string for `None`, returns `obj.data`
    when available, falls back to `_repr_html_()` when it returns a truthy value,
    and otherwise returns the string representation of `obj`.

    Args:
        obj: Object that may contain or render HTML content.

    Returns:
        Extracted HTML content, an empty string, or the string representation of
        `obj`.
    """
    if obj is None:
        return ""
    if hasattr(obj, "data"):
        return obj.data
    if hasattr(obj, "_repr_html_"):
        data = obj._repr_html_()
        if data:
            return data
    return str(obj)


def save_bertviz_html(kind: str, att_data: Mapping[str, Any], output_path: Path) -> Path:
    """Save a BERTViz head or model view as an external HTML file.

    The function renders either `head_view` or `model_view` using attention data
    and token strings from `att_data`, extracts the returned HTML, and writes it
    to `output_path`. Parent directories are created automatically when needed.

    Args:
        kind: Visualization type to save. Must be `"head"` or `"model"`.
        att_data: Mapping containing `attentions` and `tokens` entries compatible
            with BERTViz.
        output_path: Destination path for the generated HTML file.

    Returns:
        The normalized output path used to write the HTML file.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if kind == "head":
        html_obj = head_view(att_data["attentions"], att_data["tokens"], html_action="return")
    elif kind == "model":
        html_obj = model_view(att_data["attentions"], att_data["tokens"], html_action="return")
    else:
        raise ValueError("kind debe ser 'head' o 'model'.")

    html = _html_data(html_obj)
    if not html.strip():
        raise RuntimeError("BERTViz no devolvió HTML. Revisa la versión instalada de bertviz.")

    output_path.write_text(html, encoding="utf-8")
    return output_path


def show_html_file(path: Path, width: str = "100%", height: int = 720) -> None:
    """Display an external HTML file in a notebook.

    The function displays a link to open the HTML file in a new browser tab and
    embeds the same file in an iframe. The HTML file content is not read or
    embedded directly into the notebook.

    Args:
        path: Path to the external HTML file.
        width: Width passed to the displayed iframe.
        height: Height passed to the displayed iframe, in pixels.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    display(HTML(f'<p><a href="{path.as_posix()}" target="_blank">Abrir visualización en pestaña nueva</a></p>'))
    display(IFrame(src=path.as_posix(), width=width, height=height))

## Predicción del documento

In [7]:
text = get_text(TEXT_PATH, PASTED_TEXT)
result = predict_document(text, max_length=MAX_LENGTH, stride=STRIDE, batch_size=BATCH_SIZE)

print("Número de chunks:", len(result["chunks"]))
print("Autor predicho:", result["doc_pred_label"])
pretty_probabilities(result["doc_probs"])

Número de chunks: 809
Autor predicho: arthur_conan_doyle


,label,probability
0,arthur_conan_doyle,0.933985
1,richard_austin_freeman,0.027931
2,wilkie_collins,0.015046
3,gilbert_keith_chesterton,0.011932
4,anna_katharine_green,0.005965
5,arthur_morrison,0.005139


## *Chunks* candidatos para inspección

Seleccionamos los *chunks* con mayor probabilidad para la clase predicha a nivel de documento.

In [8]:
doc_pred_label = result["doc_pred_label"]
target_prob_col = f"prob_{doc_pred_label}"

top_chunk_ids = (
    result["chunk_df"].sort_values(target_prob_col, ascending=False).head(TOP_CHUNKS_TO_INSPECT)["chunk_id"].tolist()
)

print("Chunks seleccionados:", top_chunk_ids)
result["chunk_df"].loc[
    result["chunk_df"]["chunk_id"].isin(top_chunk_ids),
    ["chunk_id", "pred_label", "pred_confidence", "n_tokens", "text_preview"],
]

Chunks seleccionados: [625]


,chunk_id,pred_label,pred_confidence,n_tokens,text_preview
625,625,arthur_conan_doyle,0.999959,128,and clear against the silvered stones. The ag...


## Preparar atención para un *chunk*

La clave para que BERTViz sea manejable es recortar el *chunk* antes de pedir las atenciones.

In [9]:
chunk_id = top_chunk_ids[0]
chunk = trim_chunk_for_viz(result["chunks"][chunk_id], max_tokens=MAX_TOKENS_FOR_BERTVIZ)
att_data = get_chunk_attention(chunk)

print("Chunk:", chunk_id)
print("Tokens visualizados:", len(att_data["tokens"]))
print("Predicción del chunk:", att_data["pred_label"])
print("Primeros tokens:", att_data["tokens"][:40])
pretty_probabilities(att_data["probs"])

Chunk: 625
Tokens visualizados: 48
Predicción del chunk: arthur_conan_doyle
Primeros tokens: ['<s>', 'Ġand', 'Ġclear', 'Ġagainst', 'Ġthe', 'Ġsil', 'vered', 'Ġstones', '.', 'ĠThe', 'Ġagony', 'Ġof', 'Ġthose', 'Ġcont', 'orted', 'Ġlimbs', 'Ġstruck', 'Ġme', 'Ġwith', 'Ġa', 'Ġsp', 'asm', 'Ġof', 'Ġpain', 'Ġand', 'Ġblurred', 'Ġmy', 'Ġeyes', 'Ġwith', 'Ġtears', '.', 'Ċ', 'Ċ', '"', 'We', 'Ġmust', 'Ġsend', 'Ġfor', 'Ġhelp', ',']


,label,probability
0,arthur_conan_doyle,0.999958
1,gilbert_keith_chesterton,0.000010
2,arthur_morrison,0.000009
3,anna_katharine_green,0.000008
4,wilkie_collins,0.000007
5,richard_austin_freeman,0.000007


## BERTViz `head_view`

Esta celda guarda la visualización como un HTML externo y la muestra mediante `IFrame`. El *notebook* solo almacena el enlace/iframe, y no todo el HTML que es muy pesado.

In [10]:
if SHOW_HEAD_VIEW:
    head_html = HTML_DIR / f"head_view_chunk_{chunk_id}_tok_{len(att_data['tokens'])}.html"
    if SAVE_BERTVIZ_HTML:
        save_bertviz_html("head", att_data, head_html)
        print("Guardado:", head_html.resolve())
    show_html_file(head_html, height=720)
else:
    print("SHOW_HEAD_VIEW=False")

Guardado: /content/drive/MyDrive/outputs/authorship_transformer/bertviz_html/head_view_chunk_625_tok_48.html


## BERTViz `model_view` (opcional)

`model_view` suele generar un HTML bastante más pesado. Solo lo activaremos cuando sea necesario.

In [11]:
if SHOW_MODEL_VIEW:
    model_html = HTML_DIR / f"model_view_chunk_{chunk_id}_tok_{len(att_data['tokens'])}.html"
    if SAVE_BERTVIZ_HTML:
        save_bertviz_html("model", att_data, model_html)
        print("Guardado:", model_html.resolve())
    show_html_file(model_html, height=760)
else:
    print("SHOW_MODEL_VIEW=False. Actívalo en configuración si necesitas esta vista.")

Guardado: /content/drive/MyDrive/outputs/authorship_transformer/bertviz_html/model_view_chunk_625_tok_48.html


## Inspección manual de otro *chunk*

Para ello, establecemos `MANUAL_CHUNK_ID` y ejecutamos estas dos celdas.

In [12]:
manual_chunk = trim_chunk_for_viz(result["chunks"][MANUAL_CHUNK_ID], max_tokens=MAX_TOKENS_FOR_BERTVIZ)
manual_att_data = get_chunk_attention(manual_chunk)

print("Chunk manual:", MANUAL_CHUNK_ID)
print("Tokens visualizados:", len(manual_att_data["tokens"]))
print("Predicción:", manual_att_data["pred_label"])
pretty_probabilities(manual_att_data["probs"])

Chunk manual: 0
Tokens visualizados: 48
Predicción: arthur_conan_doyle


,label,probability
0,arthur_conan_doyle,0.999913
1,anna_katharine_green,0.000028
2,gilbert_keith_chesterton,0.000025
3,arthur_morrison,0.000014
4,richard_austin_freeman,0.000012
5,wilkie_collins,0.000009


In [13]:
manual_head_html = HTML_DIR / f"head_view_chunk_{MANUAL_CHUNK_ID}_tok_{len(manual_att_data['tokens'])}.html"
save_bertviz_html("head", manual_att_data, manual_head_html)
print("Guardado:", manual_head_html.resolve())
show_html_file(manual_head_html, height=720)

Guardado: /content/drive/MyDrive/outputs/authorship_transformer/bertviz_html/head_view_chunk_0_tok_48.html


## Resumen tabular de los *chunks* seleccionados

In [14]:
summary_rows = []
for cid in top_chunk_ids:
    chunk_for_summary = trim_chunk_for_viz(result["chunks"][cid], max_tokens=MAX_TOKENS_FOR_BERTVIZ)
    att = get_chunk_attention(chunk_for_summary)
    summary_rows.append(
        {
            "chunk_id": cid,
            "pred_label": att["pred_label"],
            "pred_confidence": float(np.max(att["probs"])),
            "n_tokens_visualized": len(att["tokens"]),
            "preview": decode_chunk(chunk_for_summary)[:250].replace("", " "),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df

,chunk_id,pred_label,pred_confidence,n_tokens_visualized,preview
0,625,arthur_conan_doyle,0.999958,48,a n d c l e a r a g a i n s t t h e ...


## Guardar resumen


In [15]:
save_dir = Path(MODEL_DIR) / "explanations_bertviz_light"
save_dir.mkdir(parents=True, exist_ok=True)

summary = {
    "model_dir": str(MODEL_DIR),
    "text_path": str(TEXT_PATH),
    "predicted_label": result["doc_pred_label"],
    "document_probabilities": {id2label[i]: float(result["doc_probs"][i]) for i in range(len(result["doc_probs"]))},
    "top_chunks_inspected": top_chunk_ids,
    "max_tokens_for_bertviz": MAX_TOKENS_FOR_BERTVIZ,
    "html_dir": str(HTML_DIR),
}

with open(save_dir / "bertviz_summary_light.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

summary_df.to_csv(save_dir / "bertviz_chunks_summary_light.csv", index=False)
print("Resumen guardado en:", save_dir.resolve())

Resumen guardado en: /content/drive/MyDrive/outputs/authorship_transformer/explanations_bertviz_light
